Claude Code 的设计哲学强调：Agent 的工具集应当遵循“少而精”的原则，优先保留具备通用能力的工具。这是因为工具数量的膨胀会不可避免地挤占宝贵的上下文窗口，进而影响模型的推理质量与响应效率。

claude code在架构方面进行了两阶段的优化(渐进式加载机制 Progressive Disclosure)：
1. tool 

当注册的工具定义超过上下文10%阈值时，会触发，不再一次性将所有工具的完整定义注入上下文，而是仅加载工具的简要名称与描述，只有当LLM判断确实需要某个工具时，才会加载这个工具的详细的参数、调用说明。按需获取，大幅降低了常驻上下文的开销。详见下面cell

2. skills

Skills 同样采用了渐进式加载的设计思路。

- 第一层，元数据(yaml frontmatter) 始终加载，成本每个skill约100个token；
```
---
name: pdf-processing
description: 提取PDF文本和表格，填写表单，合并文档。当用户提到PDF、表单或文档提取时使用。
---
```
- 第二层，指令内容（触发时才加载）

当用户的请求匹配某个技能的描述时，Claude 才会通过 bash 命令读取这个 SKILL.md 文件，这部分内容才真正进入上下文窗口

- 第三层：资源与代码（按需加载）

目录里还可以打包额外材料，比如：reference example scripts/ 等，Claude 只在被引用时才访问这些文件。


> 这里tool的描述有问题，
https://code.claude.com/docs/en/mcp#scale-with-mcp-tool-search
去专栏中把这里comment出来。
老师说的是 ENABLE_TOOL_SEARCH=auto 这种情况。但其实，默认行为是 `All MCP tools deferred and loaded on demand. `

![image.png](imgs/ENABLE_TOOL_SEARCH.png)

补充 tool的渐进式加载。

被标记为延迟加载的工具不会一开始就进入 Claude 的上下文。

模型需要用工具时，主动调用这个搜索工具（内置支持基于正则或 BM25 的检索），搜到匹配的工具后，该工具的完整定义（名称、描述、参数 schema）才被注入上下文，之后 Claude 才能正式调用它。

示例
```
{
  "tools": [
    { "type": "tool_search_tool_bm25_20251119",
      "name": "tool_search_tool_regex"
     },   // 常驻：搜索工具本身, tool search tool
    {
      "name": "github.createPullRequest",
      "description": "创建一个 Pull Request",
      "input_schema": {...},
      "defer_loading": true                          // 延迟加载
    }
    // ... 还有几百个同样标了 defer_loading: true 的工具
  ]
}
```
对于 MCP 服务器，你还可以整体延迟加载一个服务器，同时保留其中个别高频使用的工具始终加载：
```
{
  "type": "mcp_toolset",
  "mcp_server_name": "google-drive",
  "default_config": { "defer_loading": true },   // 整个服务器默认延迟加载
  "configs": {
    "search_files": { "defer_loading": false }    // 但这个常用工具保持常驻
  }
}
```

假设你连接了 5 个 MCP 服务器，共 34 个工具：

- 未触发渐进式加载时：

34 个工具的完整 JSON schema 全部塞进上下文，每轮对话都要重复携带，可能消耗 2-6 万 token 甚至更多，且很容易挤占超过 10% 的上下文预算。
- 触发渐进式加载后：

    - 上下文里只留一个"工具搜索"入口（约 500 token）+ 少数你显式保留常驻的高频工具(defer_loading": false)
    - 用户提出请求，比如"帮我在 GitHub 上创建一个 PR"
    - Claude 判断需要用到 GitHub 相关工具，于是先调用工具搜索，传入类似 "create pull request" 的检索词
    - 搜索返回匹配的 github.createPullRequest 工具引用（tool_reference）
    - 该工具的完整定义（参数、schema）这才被注入上下文
    - Claude 用这份刚加载的定义正式调用该工具完成任务

其余 33 个未被搜到的工具，全程都没有进入上下文.

原本 50+ 个 MCP 工具会在正式开始工作前就消耗约 7.7 万 token，启用 Tool Search 后只消耗约 8700 token，相当于节省了 95% 的上下文空间，整体降低了约 85% 的 token 用量。


## 金融研报系统 实现 skills
将原langgraph实现的项目中，每个业务都设计为一个skill. 方法：使用CC结合 skill-creator skill生成后，人工审阅并调整。

| Skill                       | 功能           | 具体实现业务                                         |
| --------------------------- | -------------- | ---------------------------------------------------- |
| competitor_research         | 竞争对手分析   | 通过联网搜索识别与待分析公司具体业务和市值相似的公司 |
| financial_data_collection   | 财务数据采集   | 采集指定公司三大表等财务数据                         |
| financial_ratio_calculation | 财务比率计算   | 根据采集到的财务数据，计算利润率等财务指标           |
| financial_visualization     | 财务图表生成   | 根据财务指标进行图表的绘制                           |
| valuation_modeling          | 估值与预测模型 | 根据财务指标以及行业数据进行估值计算                 |
| report_writing              | 研报写作规范   | 定义一套研报的写作规范                               |
| report_assembly             | 研报组装与输出 | 汇总前面所有 Skill 的输出，进行研报的撰写            |


项目代码见目录 `07_claudecode_skill`

对于小型项目，可以直接借助CC分析和理解项目的代码结构和功能；当针对大项目时，先针对项目生成功能文档、设计文档等，基于这些文档再去分析。 

老师推荐了一个工具 zread_cli, 一个命令行工具，通过AI阅读代码库，并生成全面的结构化wiki文档。

开发范式的变化：

去年及以前，启动一个agent项目时，‘框架先行’，即开发者先选定一个顶层脚手架(e.g. langgraph)，随后投入大量精力设计负责的图节点、处理上下文传递和编排循环逻辑，最后才是编写具体的业务工具。

今年的最佳实践是**业务先行**。开发者的首要任务不再是挑选框架，而是审视业务本身：是否存在固定的业务流程？这些流程是否可以被提炼并封装为 Skills？**一旦完成业务逻辑的 Skill 化，后续的开发便变得极为轻量。**

